# Robust fine-tuning of BEATs for a new setup

Two ideas from the literature, tested on BEATs fine-tuning:

- **SFC**, softened frequency-wise centering (Primus and Widmer, 2023, arXiv:2306.11764): before the
  encoder, each window's log-mel spectrogram loses a fraction `λ` of its mean over time,
  `x - λ · mean_t(x)`. It removes a fixed filter (the setup), but also part of the material's spectral colour.
- **LP-FT and WiSE-FT** (Kumar et al., ICLR 2022; Wortsman et al., CVPR 2022): fit the head with the encoder
  frozen, then fine-tune everything with a small learning rate; then average the weights of the probe and
  the fine-tuned model. Both aim to keep the pretrained features that plain fine-tuning distorts.

Variants are chosen on the two development transfers, Cluster → LMT108 and LMT108 → Cluster. Only the
chosen variant's SENS3 score is the number to report; SENS3 was inspected before, so it stays exploratory.

Same data, folds, weights and scores as `pretrained_encoders.ipynb` (with duplicate materials grouped);
the consistency penalty is left out. Run alone with Python 3.12+ and a GPU. `QUICK = True` checks the
pipeline (one window per recording, one fold, 30 updates); results go to
`analysis/classification/robust_finetuning/` (`robust_finetuning_quick/`). Finished trainings are reused;
start a fresh results folder after changing settings.

In [ ]:
QUICK = True

DRIVE_FILE_ID = "1NmRCDqvGvmNwQGC53IzTu9jJRaI8Newk"

INPUTS = ["audio", "audio + acceleration"]
# Variant -> (SFC weight λ, linear probe before fine-tuning)
VARIANTS = {
    "FT": (0.0, False),
    "FT + SFC 0.5": (0.5, False),
    "FT + SFC 0.9": (0.9, False),
    "FT + SFC 1.0": (1.0, False),
    "LP-FT": (0.0, True),
}
# Weight of the fine-tuned model in WiSE-FT, evaluated on every LP-FT run (0 = the frozen probe alone)
WISE_ALPHAS = [0.0, 0.25, 0.5, 0.75]
DEVELOPMENT_TESTS = ["Cluster → LMT108", "LMT108 → Cluster"]

N_UPDATES = 1500
N_PROBE_UPDATES = 300
BATCH_SIZE = 96
ACCUMULATION_STEPS = 4  # 8 GB GPU; 2 is enough with 12-16 GB
ENCODER_LR = 2e-5
HEAD_LR = 1e-3
WAVE_RMS = 0.1

## Setup: packages, data and model

In [ ]:
import importlib.util
import subprocess
import sys

PACKAGES = {
    "numpy": "numpy",
    "torch": "torch",
    "torchaudio": "torchaudio",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("installing", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

    import site

    site.addsitedir(site.getusersitepackages())
    importlib.invalidate_caches()

In [ ]:
import tarfile
import urllib.request
from pathlib import Path

ROOT = Path.cwd()
DATA_FILES = [
    "analysis/classification/waveforms.npz",
    "analysis/classification/sens3_waveforms.npz",
    "analysis/signal_processing/recordings.csv",
    "analysis/classification/sens3_recordings.csv",
]


def download_from_drive(file_id, path):

    url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"
    partial = path.with_suffix(".part")
    done = partial.stat().st_size if partial.exists() else 0
    request = urllib.request.Request(url, headers={"Range": f"bytes={done}-"} if done else {})
    with urllib.request.urlopen(request) as response, open(partial, "ab" if done else "wb") as out:
        if done and response.status != 206:
            out.truncate(0)
            done = 0
        assert "html" not in response.headers.get("Content-Type", ""), "Drive sent a web page: is the file shared?"
        total = done + int(response.headers.get("Content-Length", 0))
        while chunk := response.read(1 << 22):
            out.write(chunk)
            done += len(chunk)
            print(f"\r{done / 2**30:.2f} / {total / 2**30:.2f} GB", end="", flush=True)
    print()
    partial.rename(path)


if not all((ROOT / name).exists() for name in DATA_FILES):
    archive_path = ROOT / "data.tar"
    if not archive_path.exists():
        download_from_drive(DRIVE_FILE_ID, archive_path)
    with tarfile.open(archive_path) as archive:
        archive.extractall(ROOT, members=[archive.getmember(name) for name in DATA_FILES], filter="data")
    archive_path.unlink()
print("data ready in", ROOT)

MODELS = ROOT / "models"
BEATS_DIR = MODELS / "beats"
BEATS_DIR.mkdir(parents=True, exist_ok=True)
BEATS_CODE = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"

BEATS_WEIGHTS = "https://huggingface.co/alibabasglab/beats/resolve/main/BEATs_iter3_plus_AS2M.pt"
for name in ["BEATs.py", "backbone.py", "modules.py"]:
    if not (BEATS_DIR / name).exists():
        urllib.request.urlretrieve(BEATS_CODE + name, BEATS_DIR / name)
if not (BEATS_DIR / "BEATs_iter3_plus_AS2M.pt").exists():
    print("downloading BEATs weights (360 MB)...")
    urllib.request.urlretrieve(BEATS_WEIGHTS, BEATS_DIR / "BEATs_iter3_plus_AS2M.pt")
print("BEATs ready in", BEATS_DIR)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torchaudio
from torch import nn

sys.path.insert(0, str(BEATS_DIR))
from BEATs import BEATs, BEATsConfig  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {properties.name}, {properties.total_memory / 2**30:.0f} GB")
else:
    print("no GPU: the embeddings will be slow and the fine-tuning far too slow")

AMP_DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else torch.float16

## Evaluation

Five surface folds per dataset, stratified by family, duplicate materials grouped (seed 42). Training
weights balance (setup, family) groups. Balanced accuracy with 95% surface-bootstrap intervals; chance 0.25.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold

FAMILIES = ["wood", "metal", "textile", "polymer"]
N_FOLDS = 5
CHANCE = 1 / len(FAMILIES)
OUT = ROOT / "analysis/classification"
SAME_MATERIAL = {"13": "8", "104": "88", "G5SolidRubberPlateVersion2": "G5SolidRubberPlateVersion1", "G4CoreBeech": "G4Beech_0"}


def surface_folds(meta):
    surfaces = meta[["surface", "family"]].drop_duplicates().reset_index(drop=True)
    groups = surfaces.surface.replace(SAME_MATERIAL)
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    fold_of_surface = {}
    for fold, (_, test) in enumerate(splitter.split(surfaces.surface, surfaces.family, groups)):
        for surface in surfaces.surface.iloc[test]:
            fold_of_surface[surface] = fold
    return meta.surface.map(fold_of_surface).to_numpy()


def folds(recordings):
    fold = pd.Series(0, index=recordings.index)
    for dataset in recordings.dataset.unique():
        in_dataset = recordings.dataset == dataset
        fold[in_dataset] = surface_folds(recordings[in_dataset])
    return fold.to_numpy()


def group_weights(recordings):
    weights = 1 / recordings.groupby(["dataset", "family"]).family.transform("size").to_numpy()
    return weights / weights.mean()


def mean_recall(confusion):
    return np.mean(np.diag(confusion) / confusion.sum(axis=1))


def score(recordings, predicted):
    predicted = np.asarray(predicted)
    confusions = {}
    for surface in recordings.surface.unique():
        on_surface = (recordings.surface == surface).to_numpy()
        confusions[surface] = confusion_matrix(recordings.family[on_surface], predicted[on_surface], labels=FAMILIES)

    family_of_surface = recordings.groupby("surface").family.first()
    rng = np.random.default_rng(42)
    draws = []
    for _ in range(2000):
        resampled = np.zeros((len(FAMILIES), len(FAMILIES)))
        for family in FAMILIES:
            surfaces = family_of_surface.index[family_of_surface == family]
            for surface in rng.choice(surfaces, size=len(surfaces)):
                resampled += confusions[surface]
        draws.append(mean_recall(resampled))

    low, high = np.percentile(draws, [2.5, 97.5])
    return {"balanced accuracy": mean_recall(sum(confusions.values())), "low": low, "high": high}


def plot_scores(scores, groups, bars, panels, title, name):
    panel_names = scores[panels].unique()
    bar_names = scores[bars].unique()
    width = 0.8 / len(bar_names)

    fig, axes = plt.subplots(1, len(panel_names), figsize=(5 * len(panel_names), 4), sharey=True, squeeze=False)
    for ax, panel in zip(axes[0], panel_names, strict=True):
        in_panel = scores[scores[panels] == panel]
        group_names = in_panel[groups].unique()
        for k, bar in enumerate(bar_names):
            rows = in_panel[in_panel[bars] == bar].set_index(groups).reindex(group_names)
            x = np.arange(len(group_names)) + (k - (len(bar_names) - 1) / 2) * width
            below = rows["balanced accuracy"] - rows.low
            above = rows.high - rows["balanced accuracy"]
            ax.bar(x, rows["balanced accuracy"], width, color=f"C{k}", label=bar)
            ax.errorbar(x, rows["balanced accuracy"], yerr=[below, above], fmt="none", ecolor="black", capsize=2)
        ax.axhline(CHANCE, color="black", linestyle=":", linewidth=1)
        ax.set_xticks(range(len(group_names)), group_names, rotation=20)
        ax.set(title=panel, ylim=(0, 1))

    axes[0, 0].set_ylabel("balanced accuracy, 95% interval\n(dotted line: chance)")
    axes[0, -1].legend(title=bars, bbox_to_anchor=(1, 1), loc="upper left")
    fig.suptitle(title)
    fig.savefig(RESULTS / name, dpi=130, bbox_inches="tight")
    print("saved", RESULTS / name)


RESULTS = OUT / ("robust_finetuning_quick" if QUICK else "robust_finetuning")
RESULTS.mkdir(parents=True, exist_ok=True)

## Data

Unit-RMS 0.5 s windows: audio at 16 kHz, acceleration projected onto its dominant axis and resampled
from 8 to 16 kHz. Recording tables define the order of the waveform arrays.

In [ ]:
recordings = pd.read_csv(ROOT / "analysis/signal_processing/recordings.csv", dtype={"surface": str})
sens3 = pd.read_csv(OUT / "sens3_recordings.csv", dtype={"surface": str})
sens3["dataset"] = "sens3"


def load_waves(name):
    cache = np.load(OUT / name)
    recording_of_window = cache["recording"]
    keep = np.ones(len(recording_of_window), bool)
    if QUICK:
        keep[1:] = recording_of_window[1:] != recording_of_window[:-1]
    accel_8k = cache["accel"][keep]
    accel = torch.empty((len(accel_8k), 2 * accel_8k.shape[1]), dtype=torch.float16)
    for start in range(0, len(accel_8k), 4096):
        piece = torch.from_numpy(accel_8k[start : start + 4096].astype(np.float32))
        accel[start : start + 4096] = torchaudio.functional.resample(piece, 8000, 16000).half()
    waves = {"audio": torch.from_numpy(cache["audio"][keep]), "accel": accel}
    return waves, recording_of_window[keep]


waves, recording_of_window = load_waves("waveforms.npz")
sens3_waves, sens3_recording_of_window = load_waves("sens3_waveforms.npz")
SIGNALS_OF_INPUT = {"audio": ["audio"], "acceleration": ["accel"], "audio + acceleration": ["audio", "accel"]}

fold = folds(recordings)
in_cluster = (recordings.dataset == "cluster").to_numpy()
print(
    f"{len(recordings)} recordings, {len(recording_of_window)} windows; "
    f"SENS3: {len(sens3)} chunks, {len(sens3_recording_of_window)} windows"
)

## Model and training

BEATs iter3+ with a dropout + linear head on the mean token. SFC acts on BEATs' own log-mel
features (time × 128 mel bands), right after its preprocessing. Each batch draws random training
recordings and one random window of each; the cross-entropy is weighted per (setup, family) group.

- **FT**: encoder and head together, encoder at `ENCODER_LR`, head at `HEAD_LR`.
- **LP-FT**: `N_PROBE_UPDATES` on the head alone (frozen encoder), then `N_UPDATES` on everything at
  `ENCODER_LR`, so the probe's head is not overwritten at the start.
- **WiSE-FT**: weights `(1 - α) · probe + α · LP-FT`, evaluated without further training.

In [ ]:
def load_beats(centering):
    checkpoint = torch.load(BEATS_DIR / "BEATs_iter3_plus_AS2M.pt", map_location="cpu", weights_only=True)
    model = BEATs(BEATsConfig(checkpoint["cfg"]))
    model.load_state_dict(checkpoint["model"])

    preprocess = model.preprocess

    def centered_preprocess(source, **kwargs):
        with torch.autocast("cuda", enabled=False):
            fbank = preprocess(source.float(), **kwargs)
        return fbank - centering * fbank.mean(dim=1, keepdim=True)

    model.preprocess = centered_preprocess
    return model.to(DEVICE)


def beats_embedding(model, x):
    features, _ = model.extract_features(x.float() * WAVE_RMS)
    return features.mean(dim=1)


class FineTunedBEATs(nn.Module):
    def __init__(self, signal_names, centering):
        super().__init__()
        self.signal_names = signal_names
        self.encoder = load_beats(centering)
        width = self.encoder.cfg.encoder_embed_dim * len(signal_names)
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(width, len(FAMILIES)))

    def forward(self, batch):
        return self.head(torch.cat([beats_embedding(self.encoder, batch[name]) for name in self.signal_names], dim=1))


first_window = np.searchsorted(recording_of_window, np.arange(len(recordings)))
n_windows = np.bincount(recording_of_window, minlength=len(recordings))
family_index = recordings.family.map({name: k for k, name in enumerate(FAMILIES)}).to_numpy()


def random_windows(recs, rng):
    return first_window[recs] + rng.integers(0, n_windows[recs])


def batch_of(windows, signal_names, signal_waves=None):
    signal_waves = waves if signal_waves is None else signal_waves
    return {name: signal_waves[name][windows].to(DEVICE, non_blocking=True) for name in signal_names}


def train_model(model, train, n_updates, train_encoder, head_lr, seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model.encoder.requires_grad_(train_encoder)
    groups = [{"params": model.head.parameters(), "lr": head_lr}]
    if train_encoder:
        groups.append({"params": model.encoder.parameters(), "lr": ENCODER_LR})
    optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
    schedule = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[group["lr"] for group in groups], total_steps=n_updates, pct_start=0.1
    )
    scaler = torch.amp.GradScaler(enabled=torch.float16 == AMP_DTYPE)

    train_recs = np.flatnonzero(train)
    recording_weight = np.zeros(len(recordings))
    recording_weight[train] = group_weights(recordings[train])

    model.train()
    if not train_encoder:
        model.encoder.eval()
    for _ in range(n_updates):
        recs = rng.choice(train_recs, BATCH_SIZE)
        windows = random_windows(recs, rng)
        total_weight = recording_weight[recs].sum()

        optimizer.zero_grad()
        for rows in np.array_split(np.arange(BATCH_SIZE), ACCUMULATION_STEPS):
            targets = torch.as_tensor(family_index[recs[rows]], device=DEVICE)
            weights = torch.as_tensor(recording_weight[recs[rows]], device=DEVICE, dtype=torch.float32)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=DEVICE == "cuda"):
                logits = model(batch_of(windows[rows], model.signal_names)).float()
            losses = nn.functional.cross_entropy(logits, targets, reduction="none")
            scaler.scale((losses * weights).sum() / total_weight).backward()
        scaler.step(optimizer)
        scaler.update()
        schedule.step()
    model.encoder.requires_grad_(True)
    return model


def cpu_state(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


def interpolate(model, start, end, alpha):
    model.load_state_dict(
        {key: (1 - alpha) * start[key] + alpha * end[key] if end[key].is_floating_point() else end[key] for key in end}
    )


@torch.no_grad()
def classify(model, test_recs, rec_of_window, signal_waves):
    model.eval()
    windows = np.flatnonzero(np.isin(rec_of_window, test_recs))
    probabilities = []
    for start in range(0, len(windows), 256):
        batch = batch_of(windows[start : start + 256], model.signal_names, signal_waves)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=DEVICE == "cuda"):
            probabilities.append(model(batch).float().softmax(dim=1).cpu())
    probabilities = pd.DataFrame(torch.cat(probabilities).numpy(), columns=FAMILIES)
    return probabilities.groupby(rec_of_window[windows]).mean().idxmax(axis=1)


def finetune(input_name, variant, train, seed):
    """{model name: trained model state}; LP-FT also returns the probe and the WiSE-FT averages."""
    centering, linear_probe = VARIANTS[variant]
    model = FineTunedBEATs(SIGNALS_OF_INPUT[input_name], centering).to(DEVICE)
    n_updates = 30 if QUICK else N_UPDATES
    if not linear_probe:
        train_model(model, train, n_updates, train_encoder=True, head_lr=HEAD_LR, seed=seed)
        yield variant, model
        return

    train_model(model, train, 30 if QUICK else N_PROBE_UPDATES, train_encoder=False, head_lr=HEAD_LR, seed=seed)
    probe = cpu_state(model)
    train_model(model, train, n_updates, train_encoder=True, head_lr=ENCODER_LR, seed=seed)
    finetuned = cpu_state(model)
    yield variant, model
    for alpha in WISE_ALPHAS:
        interpolate(model, probe, finetuned, alpha)
        yield ("LP (frozen)" if alpha == 0 else f"WiSE-FT {alpha:g}"), model

## Runs

1. New Cluster surfaces: five folds (in-domain, to see what each variant costs on a known setup).
2. Development transfers: Cluster → LMT108 and LMT108 → Cluster (choose the variant here).
3. External test: Cluster + LMT108 → SENS3.

In [ ]:
import gc
import time

PREDICTIONS = RESULTS / "predictions.csv"
everything = np.ones(len(recordings), bool)

TRAININGS = [
    ("Cluster, new surfaces", k, in_cluster & (fold != k), in_cluster & (fold == k), "main")
    for k in range(1 if QUICK else N_FOLDS)
]
TRAININGS += [
    ("Cluster → LMT108", 0, in_cluster, ~in_cluster, "main"),
    ("LMT108 → Cluster", 0, ~in_cluster, in_cluster, "main"),
    ("Cluster + LMT108 → SENS3", 0, everything, np.ones(len(sens3), bool), "sens3"),
]

done = set()
if PREDICTIONS.exists():
    previous = pd.read_csv(PREDICTIONS)
    done = set(previous[["test", "fold", "input", "variant"]].drop_duplicates().itertuples(index=False, name=None))

for input_name in INPUTS:
    for variant in VARIANTS:
        for test_name, k, train, tested, dataset in TRAININGS:
            if (test_name, k, input_name, variant) in done:
                continue
            started = time.time()
            rec_of_window, signal_waves = (
                (sens3_recording_of_window, sens3_waves) if dataset == "sens3" else (recording_of_window, waves)
            )
            rows = []
            for model_name, model in finetune(input_name, variant, train, seed=k):
                predicted = classify(model, np.flatnonzero(tested), rec_of_window, signal_waves)
                rows += [
                    {
                        "test": test_name,
                        "fold": k,
                        "input": input_name,
                        "variant": variant,
                        "model": model_name,
                        "dataset": dataset,
                        "recording": r,
                        "predicted": p,
                    }
                    for r, p in predicted.items()
                ]
            pd.DataFrame(rows).to_csv(PREDICTIONS, mode="a", header=not PREDICTIONS.exists(), index=False)
            del model
            gc.collect()
            torch.cuda.empty_cache()
            print(f"{test_name}, fold {k}, {input_name}, {variant}: {time.time() - started:.0f} s", flush=True)

## Scores and choice

Each model is ranked by its mean balanced accuracy on the two development transfers, per input.
The SENS3 score printed for the chosen model is the number to report; the others are only context.

In [ ]:
predictions = pd.read_csv(PREDICTIONS)
rows = []
for (test_name, input_name, model_name, dataset), group in predictions.groupby(["test", "input", "model", "dataset"]):
    recs = (sens3 if dataset == "sens3" else recordings).loc[group.recording].reset_index(drop=True)
    rows.append({"input": input_name, "model": model_name, "test": test_name, **score(recs, group.predicted.to_numpy())})
scores = pd.DataFrame(rows)
scores.to_csv(RESULTS / "scores.csv", index=False)

table = scores.pivot_table(index=["input", "model"], columns="test", values="balanced accuracy")
table["development mean"] = table[DEVELOPMENT_TESTS].mean(axis=1)
chosen = table["development mean"].groupby(level="input").idxmax()
for input_name, key in chosen.items():
    sens3_score = scores[(scores.input == input_name) & (scores.model == key[1]) & scores.test.str.contains("SENS3")]
    print(f"{input_name}: chosen {key[1]}, SENS3 {sens3_score.iloc[0].to_dict() if len(sens3_score) else 'not run'}")
table.sort_values("development mean", ascending=False).round(2)

In [ ]:
plot_scores(
    scores,
    groups="input",
    bars="model",
    panels="test",
    title="BEATs fine-tuning: SFC, LP-FT and WiSE-FT",
    name="robust_finetuning.png",
)
plt.show()